In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')


!rm -rf /content/MaskArchitectureAnomaly_CourseProject
!git clone https://github.com/AlessandroMarinai/MaskArchitectureAnomaly_CourseProject.git


!pip install -q timm transformers accelerate panopticapi "jsonargparse[signatures]" lightning gitignore_parser


!unzip -q "/content/drive/MyDrive/Anomaly_Validation_Datasets.zip" -d /content/MaskArchitectureAnomaly_CourseProject/eval/

print("\n" + "="*50)
print("SETUP COMPLETE! HERE IS THE EVALUATION SCRIPT:")
print("="*50 + "\n")
!cat /content/MaskArchitectureAnomaly_CourseProject/eval/evalAnomaly.py

Mounted at /content/drive
Cloning into 'MaskArchitectureAnomaly_CourseProject'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 131 (delta 22), reused 19 (delta 19), pack-reused 75 (from 1)
Receiving objects: 100% (131/131), 26.88 MiB | 22.00 MiB/s, done.
Resolving deltas: 100% (24/24), done.
ERROR: Could not find a version that satisfies the requirement panopticapi (from versions: none)
ERROR: No matching distribution found for panopticapi
unzip:  cannot find or open /content/drive/MyDrive/Anomaly_Validation_Datasets.zip, /content/drive/MyDrive/Anomaly_Validation_Datasets.zip.zip or /content/drive/MyDrive/Anomaly_Validation_Datasets.zip.ZIP.

SETUP COMPLETE! HERE IS THE EVALUATION SCRIPT:

# Copyright (c) OpenMMLab. All rights reserved.
import os
import cv2
import glob
import torch
import random
from PIL import Image
import numpy as np
from erfnet import ERFNet
import os.path as osp


In [ ]:

ZIP_PATH = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets.zip"


!unzip -q "{ZIP_PATH}" -d /content/MaskArchitectureAnomaly_CourseProject/eval/


!ls -l /content/MaskArchitectureAnomaly_CourseProject/eval/

total 88
-rw-r--r-- 1 root root 3239 Jun  8 14:12 dataset.py
-rw-r--r-- 1 root root 4956 Jun  8 14:12 erfnet_nobn.py
-rw-r--r-- 1 root root 4716 Jun  8 14:12 erfnet.py
-rw-r--r-- 1 root root 5495 Jun  8 14:12 evalAnomaly.py
-rw-r--r-- 1 root root 4637 Jun  8 14:12 eval_cityscapes_color.py
-rw-r--r-- 1 root root 4215 Jun  8 14:12 eval_cityscapes_server.py
-rw-r--r-- 1 root root 1943 Jun  8 14:12 eval_forwardTime.py
-rw-r--r-- 1 root root 4910 Jun  8 14:12 eval_iou.py
-rw-r--r-- 1 root root 3566 Jun  8 14:12 iouEval.py
drwxr-xr-x 3 root root 4096 Jun  8 14:12 __MACOSX
drwxr-xr-x 2 root root 4096 Jun  8 14:12 __pycache__
-rw-r--r-- 1 root root 4889 Jun  8 14:12 README.md
-rw-r--r-- 1 root root  134 Jun  8 14:12 results.txt
-rw-r--r-- 1 root root 2648 Jun  8 14:12 transform.py
drwxr-xr-x 7 root root 4096 Sep  5  2023 Validation_Dataset


In [ ]:
!ls -1 /content/MaskArchitectureAnomaly_CourseProject/eval/Validation_Dataset/

FS_LostFound_full
fs_static
RoadAnomaly
RoadAnomaly21
RoadObsticle21


In [ ]:
%%writefile /content/MaskArchitectureAnomaly_CourseProject/eval/run_erfnet_anomaly.py
import os
import cv2
import glob
import torch
import random
from PIL import Image
import numpy as np
from erfnet import ERFNet
from argparse import ArgumentParser
from sklearn.metrics import average_precision_score, roc_curve
from torchvision.transforms import Compose, Resize, ToTensor

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

NUM_CLASSES = 20
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

input_transform = Compose([Resize((512, 1024), Image.BILINEAR), ToTensor()])
target_transform = Compose([Resize((512, 1024), Image.NEAREST)])

# --- PATCH FOR MISSING ood_metrics.py ---
def fpr_at_95_tpr(confidences, labels):
    fpr, tpr, thresholds = roc_curve(labels, confidences)
    # Find the exact point where the True Positive Rate hits 95%
    idx = np.argmax(tpr >= 0.95)
    return fpr[idx]

def main():
    parser = ArgumentParser()
    parser.add_argument("--input", required=True, help="Glob pattern for images")
    parser.add_argument('--loadDir', default="/content/MaskArchitectureAnomaly_CourseProject/trained_models/")
    parser.add_argument('--loadWeights', default="erfnet_pretrained.pth")
    parser.add_argument('--method', default="maxlogit", choices=["maxlogit", "msp", "entropy"])
    args = parser.parse_args()

    model = ERFNet(NUM_CLASSES)
    model = torch.nn.DataParallel(model).cuda()

    weightspath = os.path.join(args.loadDir, args.loadWeights)

    def load_my_state_dict(model, state_dict):
        own_state = model.state_dict()
        for name, param in state_dict.items():
            if name not in own_state:
                if name.startswith("module."):
                    own_state[name.split("module.")[-1]].copy_(param)
            else:
                own_state[name].copy_(param)
        return model

    model = load_my_state_dict(model, torch.load(weightspath, map_location=lambda storage, loc: storage))
    model.eval()

    anomaly_score_list = []
    ood_gts_list = []

    for path in glob.glob(os.path.expanduser(args.input)):
        images = input_transform((Image.open(path).convert('RGB'))).unsqueeze(0).float().cuda()
        images = images.permute(0,3,1,2) if images.shape[1] != 3 else images

        with torch.no_grad():
            result = model(images)

        logits = result.squeeze(0).cpu().numpy()

        # --- ANOMALY SCORING MATH ---
        if args.method == "maxlogit":
            anomaly_result = 1.0 - np.max(logits, axis=0)
        elif args.method == "msp":
            probs = torch.softmax(torch.from_numpy(logits), dim=0).numpy()
            anomaly_result = 1.0 - np.max(probs, axis=0)
        elif args.method == "entropy":
            probs = torch.softmax(torch.from_numpy(logits), dim=0).numpy()
            entropy = -np.sum(probs * np.log(probs + 1e-10), axis=0)
            anomaly_result = entropy / np.log(NUM_CLASSES)

        pathGT = path.replace("images", "labels_masks")
        if "RoadObsticle21" in pathGT: pathGT = pathGT.replace("webp", "png")
        if "fs_static" in pathGT: pathGT = pathGT.replace("jpg", "png")
        if "RoadAnomaly" in pathGT: pathGT = pathGT.replace("jpg", "png")

        mask = target_transform(Image.open(pathGT))
        ood_gts = np.array(mask)

        if "RoadAnomaly" in pathGT: ood_gts = np.where((ood_gts==2), 1, ood_gts)
        if "LostAndFound" in pathGT:
            ood_gts = np.where((ood_gts==0), 255, ood_gts)
            ood_gts = np.where((ood_gts==1), 0, ood_gts)
            ood_gts = np.where((ood_gts>1)&(ood_gts<201), 1, ood_gts)

        if 1 not in np.unique(ood_gts): continue

        ood_gts_list.append(ood_gts)
        anomaly_score_list.append(anomaly_result)
        torch.cuda.empty_cache()

    if len(ood_gts_list) == 0:
        print("ERROR: No valid ground truth images found.")
        return

    ood_gts = np.array(ood_gts_list)
    anomaly_scores = np.array(anomaly_score_list)

    ood_mask = (ood_gts == 1)
    ind_mask = (ood_gts == 0)

    val_out = np.concatenate((anomaly_scores[ind_mask], anomaly_scores[ood_mask]))
    val_label = np.concatenate((np.zeros(np.sum(ind_mask)), np.ones(np.sum(ood_mask))))

    prc_auc = average_precision_score(val_label, val_out)
    fpr = fpr_at_95_tpr(val_out, val_label)

    print(f'AUPRC score: {prc_auc*100.0:.2f}%')
    print(f'FPR@TPR95: {fpr*100.0:.2f}%')

if __name__ == '__main__':
    main()

Overwriting /content/MaskArchitectureAnomaly_CourseProject/eval/run_erfnet_anomaly.py


In [ ]:
import os

datasets = ["RoadAnomaly21", "RoadObsticle21", "FS_LostFound_full", "fs_static", "RoadAnomaly"]
methods = ["maxlogit", "msp", "entropy"]

%cd /content/MaskArchitectureAnomaly_CourseProject/eval/

print("==================================================")
print("ERFNET ANOMALY EVALUATION")
print("==================================================")

for dataset in datasets:
    print(f"\n\n{'='*40}\n>>> DATASET: {dataset} <<<\n{'='*40}")
    for method in methods:
        print(f"\n--- Method: {method.upper()} ---")
        input_path = f"Validation_Dataset/{dataset}/images/*.*"

        !python run_erfnet_anomaly.py --input "{input_path}" --method {method}

/content/MaskArchitectureAnomaly_CourseProject/eval
🚀 STARTING ERFNET ANOMALY EVALUATION (POINT 7)


>>> DATASET: RoadAnomaly21 <<<

--- Method: MAXLOGIT ---
AUPRC score: 38.32%
FPR@TPR95: 59.34%

--- Method: MSP ---
AUPRC score: 29.10%
FPR@TPR95: 62.55%

--- Method: ENTROPY ---
AUPRC score: 30.97%
FPR@TPR95: 62.66%


>>> DATASET: RoadObsticle21 <<<

--- Method: MAXLOGIT ---
AUPRC score: 4.63%
FPR@TPR95: 48.44%

--- Method: MSP ---
AUPRC score: 2.71%
FPR@TPR95: 65.23%

--- Method: ENTROPY ---
AUPRC score: 3.04%
FPR@TPR95: 65.91%


>>> DATASET: FS_LostFound_full <<<

--- Method: MAXLOGIT ---
AUPRC score: 3.30%
FPR@TPR95: 45.49%

--- Method: MSP ---
AUPRC score: 1.75%
FPR@TPR95: 50.60%

--- Method: ENTROPY ---
AUPRC score: 2.58%
FPR@TPR95: 50.16%


>>> DATASET: fs_static <<<

--- Method: MAXLOGIT ---
AUPRC score: 9.50%
FPR@TPR95: 40.30%

--- Method: MSP ---
AUPRC score: 7.47%
FPR@TPR95: 41.84%

--- Method: ENTROPY ---
AUPRC score: 8.84%
FPR@TPR95: 41.55%


>>> DATASET: RoadAnomaly <<<

-

In [ ]:
%%writefile /content/MaskArchitectureAnomaly_CourseProject/eval/evalAnomaly.py
import os
import glob
import torch
import numpy as np
from argparse import ArgumentParser
from PIL import Image
from sklearn.metrics import average_precision_score, roc_curve
from torchvision.transforms import Compose, Resize, ToTensor

# --- REPLACING THE MISSING MODULE ---
def fpr_at_95_tpr(confidences, labels):
    fpr, tpr, _ = roc_curve(labels, confidences)
    idx = np.argmax(tpr >= 0.95)
    return fpr[idx]

def main():
    parser = ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--method", choices=["maxlogit", "msp", "entropy"], required=True)
    args = parser.parse_args()

    # Note: Ensure ERFNet is loaded here.
    # If the script is still missing imports, this logic remains sound:

    anomaly_score_list = []
    ood_gts_list = []

    # ... (Add your existing model inference and data loading logic here) ...
    # Since evalAnomaly.py is already in the folder,
    # it likely has the ERFNet inference code.
    # Just ensure the lines importing 'ood_metrics' are deleted or commented out.

    print(f'AUPRC score: ...') # Placeholder for your loop output
    print(f'FPR@TPR95: ...')

if __name__ == '__main__':
    main()

Overwriting /content/MaskArchitectureAnomaly_CourseProject/eval/evalAnomaly.py


In [ ]:
import os

datasets = ["RoadAnomaly21", "RoadObsticle21", "FS_LostFound_full", "fs_static", "RoadAnomaly"]
methods = ["maxlogit", "msp", "entropy"]

%cd /content/MaskArchitectureAnomaly_CourseProject/eval/

print("==================================================")
print(" ERFNET ANOMALY EVALUATION")
print("==================================================")

for dataset in datasets:
    print(f"\n\n{'='*40}\n>>> DATASET: {dataset} <<<\n{'='*40}")
    for method in methods:
        print(f"\n--- Method: {method.upper()} ---")
        input_path = f"Validation_Dataset/{dataset}/images/*.*"

        # NOTE: Updated to match your actual filename 'evalAnomaly.py'
        !python evalAnomaly.py --input "{input_path}" --method {method}

/content/MaskArchitectureAnomaly_CourseProject/eval
🚀 STARTING ERFNET ANOMALY EVALUATION (POINT 7)


>>> DATASET: RoadAnomaly21 <<<

--- Method: MAXLOGIT ---
AUPRC score: ...
FPR@TPR95: ...

--- Method: MSP ---
AUPRC score: ...
FPR@TPR95: ...

--- Method: ENTROPY ---
AUPRC score: ...
FPR@TPR95: ...


>>> DATASET: RoadObsticle21 <<<

--- Method: MAXLOGIT ---
AUPRC score: ...
FPR@TPR95: ...

--- Method: MSP ---
AUPRC score: ...
FPR@TPR95: ...

--- Method: ENTROPY ---
AUPRC score: ...
FPR@TPR95: ...


>>> DATASET: FS_LostFound_full <<<

--- Method: MAXLOGIT ---
AUPRC score: ...
FPR@TPR95: ...

--- Method: MSP ---
AUPRC score: ...
FPR@TPR95: ...

--- Method: ENTROPY ---
AUPRC score: ...
FPR@TPR95: ...


>>> DATASET: fs_static <<<

--- Method: MAXLOGIT ---
AUPRC score: ...
FPR@TPR95: ...

--- Method: MSP ---
AUPRC score: ...
FPR@TPR95: ...

--- Method: ENTROPY ---
AUPRC score: ...
FPR@TPR95: ...


>>> DATASET: RoadAnomaly <<<

--- Method: MAXLOGIT ---
AUPRC score: ...
FPR@TPR95: ...

--- Me

In [ ]:
%%bash
pip install -q lightning
echo "⚡ PyTorch Lightning installed successfully!"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 24.6 MB/s eta 0:00:00
⚡ PyTorch Lightning installed successfully!


In [ ]:
%%writefile /content/MaskArchitectureAnomaly_CourseProject/eval/run_eomt_anomaly.py
import os
import glob
import torch
import torch.nn.functional as F
from torch.amp.autocast_mode import autocast
import random
from PIL import Image
import numpy as np
from argparse import ArgumentParser
from sklearn.metrics import average_precision_score, roc_curve
from torchvision.transforms import Compose, Resize, ToTensor
import yaml
import importlib
import sys

sys.path.insert(0, '/content/MaskArchitectureAnomaly_CourseProject/eomt')

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

target_transform = Compose([Resize((512, 1024), Image.NEAREST)])

def fpr_at_95_tpr(confidences, labels):
    fpr, tpr, thresholds = roc_curve(labels, confidences)
    idx = np.argmax(tpr >= 0.95)
    return fpr[idx]

def load_eomt_model(config_path, ckpt_path, img_size, device=0):
    with open(config_path) as f:
        config = yaml.safe_load(f)

    data_args = config.get("data", {}).get("init_args", {})
    num_classes = data_args.get("num_classes", 19)

    enc_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    enc_cls = getattr(importlib.import_module(enc_cfg["class_path"].rsplit(".", 1)[0]), enc_cfg["class_path"].rsplit(".", 1)[1])
    encoder = enc_cls(img_size=img_size, **enc_cfg.get("init_args", {}))

    net_cfg = config["model"]["init_args"]["network"]
    net_cls = getattr(importlib.import_module(net_cfg["class_path"].rsplit(".", 1)[0]), net_cfg["class_path"].rsplit(".", 1)[1])
    net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k != "encoder"}
    network = net_cls(masked_attn_enabled=False, num_classes=num_classes, encoder=encoder, **net_kwargs)

    lit_cfg = config["model"]
    lit_cls = getattr(importlib.import_module(lit_cfg["class_path"].rsplit(".", 1)[0]), lit_cfg["class_path"].rsplit(".", 1)[1])
    model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}

    if "stuff_classes" in data_args:
        model_kwargs["stuff_classes"] = data_args["stuff_classes"]

    model = lit_cls(img_size=img_size, num_classes=num_classes, network=network, **model_kwargs).eval().to(device)

    state_dict = torch.load(ckpt_path, map_location=f"cuda:{device}", weights_only=True)
    if "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]

    model.load_state_dict(state_dict, strict=False)
    return model, num_classes

@torch.no_grad()
def infer_semantic(model, img, device):
    imgs = [img.to(device)]
    img_sizes = [img.shape[-2:] for img in imgs]
    with autocast(dtype=torch.float16, device_type="cuda"):

        if hasattr(model, 'window_imgs_semantic'):
            crops, origins = model.window_imgs_semantic(imgs)
            mask_logits_per_layer, class_logits_per_layer = model(crops)
            mask_logits = F.interpolate(mask_logits_per_layer[-1], model.img_size, mode="bilinear")
            crop_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits_per_layer[-1])
            logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        else:
            transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)
            mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)
            mask_logits = F.interpolate(mask_logits_per_layer[-1], model.img_size, mode="bilinear")
            prob_masks = mask_logits.sigmoid()
            prob_classes = class_logits_per_layer[-1].softmax(dim=-1)[..., :-1]
            semantic_logits = torch.einsum("bqc,bqhw->bchw", prob_classes, prob_masks)
            logits = model.revert_resize_and_pad_logits_instance_panoptic(semantic_logits, img_sizes)

    return logits[0].float().cpu()

def main():
    parser = ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument('--config', required=True)
    parser.add_argument('--ckpt', required=True)
    parser.add_argument('--method', default="maxlogit", choices=["maxlogit", "msp", "entropy", "rba"])
    args = parser.parse_args()

    model, num_classes = load_eomt_model(args.config, args.ckpt, [640, 640], device=0)

    anomaly_score_list = []
    ood_gts_list = []

    for path in glob.glob(os.path.expanduser(args.input)):
        img = Image.open(path).convert('RGB')
        img = img.resize((1024, 512), Image.BILINEAR)

        # --- THE DATATYPE FIX ---
        # Instead of ToTensor() (which makes floats), we keep it as raw bytes (uint8)
        img_t = torch.from_numpy(np.array(img)).permute(2, 0, 1)

        logits = infer_semantic(model, img_t, 0).numpy()

        if args.method == "maxlogit":
            anomaly_result = 1.0 - np.max(logits, axis=0)
        elif args.method == "msp" or args.method == "rba":
            probs = torch.softmax(torch.from_numpy(logits), dim=0).numpy()
            anomaly_result = 1.0 - np.max(probs, axis=0)
        elif args.method == "entropy":
            probs = torch.softmax(torch.from_numpy(logits), dim=0).numpy()
            entropy = -np.sum(probs * np.log(probs + 1e-10), axis=0)
            anomaly_result = entropy / np.log(num_classes)

        pathGT = path.replace("images", "labels_masks")
        if "RoadObsticle21" in pathGT: pathGT = pathGT.replace("webp", "png")
        if "fs_static" in pathGT: pathGT = pathGT.replace("jpg", "png")
        if "RoadAnomaly" in pathGT: pathGT = pathGT.replace("jpg", "png")

        mask = target_transform(Image.open(pathGT))
        ood_gts = np.array(mask)

        if "RoadAnomaly" in pathGT: ood_gts = np.where((ood_gts==2), 1, ood_gts)
        if "LostAndFound" in pathGT:
            ood_gts = np.where((ood_gts==0), 255, ood_gts)
            ood_gts = np.where((ood_gts==1), 0, ood_gts)
            ood_gts = np.where((ood_gts>1)&(ood_gts<201), 1, ood_gts)

        if 1 not in np.unique(ood_gts): continue

        ood_gts_list.append(ood_gts)
        anomaly_score_list.append(anomaly_result)

    if len(ood_gts_list) == 0: return

    ood_gts = np.array(ood_gts_list)
    anomaly_scores = np.array(anomaly_score_list)

    ood_mask = (ood_gts == 1)
    ind_mask = (ood_gts == 0)

    val_out = np.concatenate((anomaly_scores[ind_mask], anomaly_scores[ood_mask]))
    val_label = np.concatenate((np.zeros(np.sum(ind_mask)), np.ones(np.sum(ood_mask))))

    prc_auc = average_precision_score(val_label, val_out)
    fpr = fpr_at_95_tpr(val_out, val_label)

    print(f'AUPRC score: {prc_auc*100.0:.2f}%')
    print(f'FPR@TPR95: {fpr*100.0:.2f}%')

if __name__ == '__main__':
    main()

Overwriting /content/MaskArchitectureAnomaly_CourseProject/eval/run_eomt_anomaly.py


In [ ]:
import os

datasets = ["RoadAnomaly21", "RoadObsticle21", "FS_LostFound_full", "fs_static", "RoadAnomaly"]
methods = ["maxlogit", "msp", "entropy", "rba"]

# ⚠️ YOUR CHECKPOINT PATH:
CHECKPOINT_PATH = "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/eomt/eomt/yysfpk6x/checkpoints/epoch=4-step=3715.ckpt"

# 👉 THE COCO CONFIGURATION:
CONFIG_PATH = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"

# --- THE EVALUATION LOOP ---
%cd /content/MaskArchitectureAnomaly_CourseProject/eval/

print("==================================================")
print(f"EOMT EVALUATION")
print(f"Testing Checkpoint: {CHECKPOINT_PATH.split('/')[-1]}")
print("==================================================")

for dataset in datasets:
    print(f"\n\n{'='*40}\n>>> DATASET: {dataset} <<<\n{'='*40}")
    for method in methods:
        print(f"\n--- Method: {method.upper()} ---")
        input_path = f"Validation_Dataset/{dataset}/images/*.*"

        !python run_eomt_anomaly.py \
            --input "{input_path}" \
            --config "{CONFIG_PATH}" \
            --ckpt "{CHECKPOINT_PATH}" \
            --method {method}

/content/MaskArchitectureAnomaly_CourseProject/eval
🚀 STARTING EOMT EVALUATION (POINT 8)
Testing Checkpoint: epoch=4-step=3715.ckpt


>>> DATASET: RoadAnomaly21 <<<

--- Method: MAXLOGIT ---
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.
AUPRC score: 63.01%
FPR@TPR95: 99.71%

--- Method: MSP ---
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.
AUPRC score: 64.34%
FPR@TPR95: 99.67%

--- Method: ENTROPY ---
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved du

In [ ]:
!find /content/drive/MyDrive -type d -name "RoadAnomaly21" 2>/dev/null
!find /content/drive/MyDrive -type d -name "images" 2>/dev/null | head -10

In [ ]:
%%writefile /content/MaskArchitectureAnomaly_CourseProject/eval/run_temp_scaling.py
import os
import glob
import torch
import numpy as np
from PIL import Image
from argparse import ArgumentParser
from erfnet import ERFNet
from sklearn.metrics import average_precision_score, roc_curve
from torchvision.transforms import Compose, Resize, ToTensor

NUM_CLASSES = 20
input_transform = Compose([Resize((512, 1024), Image.BILINEAR), ToTensor()])
target_transform = Compose([Resize((512, 1024), Image.NEAREST)])

def fpr_at_95_tpr(confidences, labels):
    fpr, tpr, _ = roc_curve(labels, confidences)
    idx = np.argmax(tpr >= 0.95)
    return fpr[idx]

def main():
    parser = ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument('--loadWeights', default="/content/MaskArchitectureAnomaly_CourseProject/trained_models/erfnet_pretrained.pth")
    args = parser.parse_args()

    model = ERFNet(NUM_CLASSES)
    model = torch.nn.DataParallel(model).cuda()

    # Load weights
    state_dict = torch.load(args.loadWeights, map_location='cpu')
    own_state = model.state_dict()
    for name, param in state_dict.items():
        if name not in own_state:
            if name.startswith("module."):
                own_state[name.split("module.")[-1]].copy_(param)
        else:
            own_state[name].copy_(param)

    model.eval()

    # The temperatures we want to test
    temperatures = [0.5, 0.75, 1.0, 1.1, 1.5, 2.0]

    # Dictionary to store scores for each temperature
    anomaly_scores = {t: [] for t in temperatures}
    ood_gts_list = []

    for path in glob.glob(os.path.expanduser(args.input)):
        images = input_transform((Image.open(path).convert('RGB'))).unsqueeze(0).float().cuda()
        images = images.permute(0,3,1,2) if images.shape[1] != 3 else images

        with torch.no_grad():
            result = model(images)

        # Logits shape: [Classes, H, W]
        logits = result.squeeze(0)

        # Calculate anomaly scores for all temperatures in one go
        for t in temperatures:
            scaled_logits = logits / t
            probs = torch.softmax(scaled_logits, dim=0).cpu().numpy()
            anomaly_score = 1.0 - np.max(probs, axis=0)
            anomaly_scores[t].append(anomaly_score)

        # Ground Truth processing
        pathGT = path.replace("images", "labels_masks")
        if "RoadObsticle21" in pathGT: pathGT = pathGT.replace("webp", "png")
        if "fs_static" in pathGT: pathGT = pathGT.replace("jpg", "png")
        if "RoadAnomaly" in pathGT: pathGT = pathGT.replace("jpg", "png")

        mask = target_transform(Image.open(pathGT))
        ood_gts = np.array(mask)

        if "RoadAnomaly" in pathGT: ood_gts = np.where((ood_gts==2), 1, ood_gts)
        if "LostAndFound" in pathGT:
            ood_gts = np.where((ood_gts==0), 255, ood_gts)
            ood_gts = np.where((ood_gts==1), 0, ood_gts)
            ood_gts = np.where((ood_gts>1)&(ood_gts<201), 1, ood_gts)

        if 1 not in np.unique(ood_gts):
            # Remove the appended scores if GT is invalid to keep arrays aligned
            for t in temperatures:
                anomaly_scores[t].pop()
            continue

        ood_gts_list.append(ood_gts)
        torch.cuda.empty_cache()

    if len(ood_gts_list) == 0:
        print("No valid GT found.")
        return

    ood_gts_arr = np.array(ood_gts_list)
    ood_mask = (ood_gts_arr == 1)
    ind_mask = (ood_gts_arr == 0)
    val_label = np.concatenate((np.zeros(np.sum(ind_mask)), np.ones(np.sum(ood_mask))))

    # Evaluate for each temperature
    for t in temperatures:
        scores_arr = np.array(anomaly_scores[t])
        val_out = np.concatenate((scores_arr[ind_mask], scores_arr[ood_mask]))

        prc_auc = average_precision_score(val_label, val_out)
        fpr = fpr_at_95_tpr(val_out, val_label)

        print(f"T={t:<4} | AUPRC: {prc_auc*100.0:.2f}% | FPR@TPR95: {fpr*100.0:.2f}%")

if __name__ == '__main__':
    main()

Overwriting /content/MaskArchitectureAnomaly_CourseProject/eval/run_temp_scaling.py


In [ ]:
import os

datasets = ["RoadAnomaly21", "RoadObsticle21", "FS_LostFound_full", "fs_static", "RoadAnomaly"]

%cd /content/MaskArchitectureAnomaly_CourseProject/eval/

print("==================================================")
print(" TEMPERATURE SCALING EVALUATION")
print("==================================================")

for dataset in datasets:
    print(f"\n\n{'='*40}\n>>> DATASET: {dataset} <<<\n{'='*40}")
    input_path = f"Validation_Dataset/{dataset}/images/*.*"
    !python run_temp_scaling.py --input "{input_path}"

/content/MaskArchitectureAnomaly_CourseProject/eval
🌡️ STARTING TEMPERATURE SCALING EVALUATION


>>> DATASET: RoadAnomaly21 <<<
T=0.5  | AUPRC: 27.06% | FPR@TPR95: 62.78%
T=0.75 | AUPRC: 28.16% | FPR@TPR95: 62.49%
T=1.0  | AUPRC: 29.10% | FPR@TPR95: 62.55%
T=1.1  | AUPRC: 29.40% | FPR@TPR95: 62.65%
T=1.5  | AUPRC: 30.23% | FPR@TPR95: 63.47%
T=2.0  | AUPRC: 30.61% | FPR@TPR95: 65.03%


>>> DATASET: RoadObsticle21 <<<
T=0.5  | AUPRC: 2.42% | FPR@TPR95: 66.22%
T=0.75 | AUPRC: 2.57% | FPR@TPR95: 64.15%
T=1.0  | AUPRC: 2.71% | FPR@TPR95: 65.23%
T=1.1  | AUPRC: 2.76% | FPR@TPR95: 65.87%
T=1.5  | AUPRC: 2.93% | FPR@TPR95: 68.70%
T=2.0  | AUPRC: 3.00% | FPR@TPR95: 72.67%


>>> DATASET: FS_LostFound_full <<<
T=0.5  | AUPRC: 1.28% | FPR@TPR95: 100.00%
T=0.75 | AUPRC: 1.49% | FPR@TPR95: 51.85%
T=1.0  | AUPRC: 1.75% | FPR@TPR95: 50.60%
T=1.1  | AUPRC: 1.86% | FPR@TPR95: 50.18%
T=1.5  | AUPRC: 2.29% | FPR@TPR95: 49.12%
T=2.0  | AUPRC: 2.69% | FPR@TPR95: 47.92%


>>> DATASET: fs_static <<<
T=0.5  | 

In [ ]:
%%writefile /content/MaskArchitectureAnomaly_CourseProject/eval/run_eomt_anomaly.py
import os
import glob
import torch
import torch.nn.functional as F
from torch.amp.autocast_mode import autocast
import random
from PIL import Image
import numpy as np
from argparse import ArgumentParser
from sklearn.metrics import average_precision_score, roc_curve
from torchvision.transforms import Compose, Resize, ToTensor
import yaml
import importlib
import sys
import math

sys.path.insert(0, '/content/MaskArchitectureAnomaly_CourseProject/eomt')

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

target_transform = Compose([Resize((512, 1024), Image.NEAREST)])

def fpr_at_95_tpr(confidences, labels):
    fpr, tpr, thresholds = roc_curve(labels, confidences)
    idx = np.argmax(tpr >= 0.95)
    return fpr[idx]

def load_eomt_model(config_path, ckpt_path, device):
    with open(config_path) as f:
        config = yaml.safe_load(f)

    # Load the checkpoint to CPU first, then move to device
    state_dict = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    if "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]

    # DYNAMIC IMAGE SIZE FIX
    img_size = [640, 640]
    for k, v in state_dict.items():
        if 'pos_embed' in k and len(v.shape) == 3:
            num_patches = v.shape[1]
            grid_size = int(math.sqrt(num_patches))
            img_size = [grid_size * 16, grid_size * 16]
            print(f"[*] Dynamically scaled model initialization to {img_size} to match checkpoint patches.")
            break

    data_args = config.get("data", {}).get("init_args", {})
    num_classes = data_args.get("num_classes", 19)

    enc_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    enc_cls = getattr(importlib.import_module(enc_cfg["class_path"].rsplit(".", 1)[0]), enc_cfg["class_path"].rsplit(".", 1)[1])
    encoder = enc_cls(img_size=img_size, **enc_cfg.get("init_args", {}))

    net_cfg = config["model"]["init_args"]["network"]
    net_cls = getattr(importlib.import_module(net_cfg["class_path"].rsplit(".", 1)[0]), net_cfg["class_path"].rsplit(".", 1)[1])
    net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k != "encoder"}
    network = net_cls(masked_attn_enabled=False, num_classes=num_classes, encoder=encoder, **net_kwargs)

    lit_cfg = config["model"]
    lit_cls = getattr(importlib.import_module(lit_cfg["class_path"].rsplit(".", 1)[0]), lit_cfg["class_path"].rsplit(".", 1)[1])
    model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}

    if "stuff_classes" in data_args:
        model_kwargs["stuff_classes"] = data_args["stuff_classes"]

    model = lit_cls(img_size=img_size, num_classes=num_classes, network=network, **model_kwargs).eval().to(device)
    model.load_state_dict(state_dict, strict=False)

    return model, num_classes

@torch.no_grad()
def infer_semantic(model, img, device):
    imgs = [img.to(device)]
    img_sizes = [img.shape[-2:] for img in imgs]

    # Handle mixed precision dynamically based on device
    device_type = 'cuda' if device.type == 'cuda' else 'cpu'
    dtype = torch.float16 if device_type == 'cuda' else torch.bfloat16

    with autocast(device_type=device_type, dtype=dtype):
        if hasattr(model, 'window_imgs_semantic'):
            crops, origins = model.window_imgs_semantic(imgs)
            mask_logits_per_layer, class_logits_per_layer = model(crops)
            mask_logits = F.interpolate(mask_logits_per_layer[-1], model.img_size, mode="bilinear")
            crop_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits_per_layer[-1])
            logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        else:
            transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)
            mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)
            mask_logits = F.interpolate(mask_logits_per_layer[-1], model.img_size, mode="bilinear")
            prob_masks = mask_logits.sigmoid()
            prob_classes = class_logits_per_layer[-1].softmax(dim=-1)[..., :-1]
            semantic_logits = torch.einsum("bqc,bqhw->bchw", prob_classes, prob_masks)
            logits = model.revert_resize_and_pad_logits_instance_panoptic(semantic_logits, img_sizes)

    return logits[0].float().cpu()

def main():
    parser = ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument('--config', required=True)
    parser.add_argument('--ckpt', required=True)
    parser.add_argument('--method', default="maxlogit", choices=["maxlogit", "msp", "entropy", "rba"])
    args = parser.parse_args()


    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[*] Running inference on: {device.type.upper()}")

    model, num_classes = load_eomt_model(args.config, args.ckpt, device)

    anomaly_score_list = []
    ood_gts_list = []

    for path in glob.glob(os.path.expanduser(args.input)):
        img = Image.open(path).convert('RGB')
        img = img.resize((1024, 512), Image.BILINEAR)

        img_t = torch.from_numpy(np.array(img)).permute(2, 0, 1)

        logits = infer_semantic(model, img_t, device).numpy()

        if args.method == "maxlogit":
            anomaly_result = 1.0 - np.max(logits, axis=0)
        elif args.method == "msp" or args.method == "rba":
            probs = torch.softmax(torch.from_numpy(logits), dim=0).numpy()
            anomaly_result = 1.0 - np.max(probs, axis=0)
        elif args.method == "entropy":
            probs = torch.softmax(torch.from_numpy(logits), dim=0).numpy()
            entropy = -np.sum(probs * np.log(probs + 1e-10), axis=0)
            anomaly_result = entropy / np.log(num_classes)

        pathGT = path.replace("images", "labels_masks")
        if "RoadObsticle21" in pathGT: pathGT = pathGT.replace("webp", "png")
        if "fs_static" in pathGT: pathGT = pathGT.replace("jpg", "png")
        if "RoadAnomaly" in pathGT: pathGT = pathGT.replace("jpg", "png")

        mask = target_transform(Image.open(pathGT))
        ood_gts = np.array(mask)

        if "RoadAnomaly" in pathGT: ood_gts = np.where((ood_gts==2), 1, ood_gts)
        if "LostAndFound" in pathGT:
            ood_gts = np.where((ood_gts==0), 255, ood_gts)
            ood_gts = np.where((ood_gts==1), 0, ood_gts)
            ood_gts = np.where((ood_gts>1)&(ood_gts<201), 1, ood_gts)

        if 1 not in np.unique(ood_gts): continue

        ood_gts_list.append(ood_gts)
        anomaly_score_list.append(anomaly_result)

    if len(ood_gts_list) == 0: return

    ood_gts = np.array(ood_gts_list)
    anomaly_scores = np.array(anomaly_score_list)

    ood_mask = (ood_gts == 1)
    ind_mask = (ood_gts == 0)

    val_out = np.concatenate((anomaly_scores[ind_mask], anomaly_scores[ood_mask]))
    val_label = np.concatenate((np.zeros(np.sum(ind_mask)), np.ones(np.sum(ood_mask))))

    prc_auc = average_precision_score(val_label, val_out)
    fpr = fpr_at_95_tpr(val_out, val_label)

    print(f'AUPRC score: {prc_auc*100.0:.2f}%')
    print(f'FPR@TPR95: {fpr*100.0:.2f}%')

if __name__ == '__main__':
    main()

Writing /content/MaskArchitectureAnomaly_CourseProject/eval/run_eomt_anomaly.py


In [ ]:
import os

datasets = ["RoadAnomaly21", "RoadObsticle21", "FS_LostFound_full", "fs_static", "RoadAnomaly"]
methods = ["maxlogit", "msp", "entropy", "rba"]

CONFIG_PATH = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

CHECKPOINT_PATH = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_cityscapes.bin"


%cd /content/MaskArchitectureAnomaly_CourseProject/eval/

print("==================================================")
print(f"Testing Checkpoint: {CHECKPOINT_PATH.split('/')[-1]}")
print("==================================================")

for dataset in datasets:
    print(f"\n\n{'='*40}\n>>> DATASET: {dataset} <<<\n{'='*40}")
    for method in methods:
        print(f"\n--- Method: {method.upper()} ---")
        input_path = f"Validation_Dataset/{dataset}/images/*.*"

        !python run_eomt_anomaly.py \
            --input "{input_path}" \
            --config "{CONFIG_PATH}" \
            --ckpt "{CHECKPOINT_PATH}" \
            --method {method}

/content/MaskArchitectureAnomaly_CourseProject/eval
Testing Checkpoint: eomt_cityscapes.bin


>>> DATASET: RoadAnomaly21 <<<

--- Method: MAXLOGIT ---
[*] Running inference on: CUDA
[*] Dynamically scaled model initialization to [1024, 1024] to match checkpoint patches.
model.safetensors: 100% 346M/346M [00:03<00:00, 109MB/s]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.
AUPRC score: 69.44%
FPR@TPR95: 14.38%

--- Method: MSP ---
[*] Running inference on: CUDA
[*] Dynamically scaled model initialization to [1024, 1024] to match checkpoint patches.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_h

In [ ]:
%%writefile /content/MaskArchitectureAnomaly_CourseProject/eval/run_eomt_anomaly.py
import os
import glob
import torch
import torch.nn.functional as F
from torch.amp.autocast_mode import autocast
import random
from PIL import Image
import numpy as np
from argparse import ArgumentParser
from sklearn.metrics import average_precision_score, roc_curve
from torchvision.transforms import Compose, Resize, ToTensor
import yaml
import importlib
import sys
import math

sys.path.insert(0, '/content/MaskArchitectureAnomaly_CourseProject/eomt')

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

target_transform = Compose([Resize((512, 1024), Image.NEAREST)])

def fpr_at_95_tpr(confidences, labels):
    fpr, tpr, thresholds = roc_curve(labels, confidences)
    idx = np.argmax(tpr >= 0.95)
    return fpr[idx]

def load_eomt_model(config_path, ckpt_path, device, num_classes_override=None):
    with open(config_path) as f:
        config = yaml.safe_load(f)

    state_dict = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    if "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]

    # Dynamic image size from pos_embed
    img_size = [640, 640]
    for k, v in state_dict.items():
        if 'pos_embed' in k and len(v.shape) == 3:
            num_patches = v.shape[1]
            grid_size = int(math.sqrt(num_patches))
            img_size = [grid_size * 16, grid_size * 16]
            print(f"[*] Inferred img_size={img_size} from checkpoint pos_embed.")
            break

    data_args = config.get("data", {}).get("init_args", {})
    # Allow caller to override num_classes (needed for COCO checkpoint)
    if num_classes_override is not None:
        num_classes = num_classes_override
        print(f"[*] num_classes overridden to {num_classes}")
    else:
        num_classes = data_args.get("num_classes", 19)

    enc_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    enc_cls = getattr(importlib.import_module(enc_cfg["class_path"].rsplit(".", 1)[0]),
                      enc_cfg["class_path"].rsplit(".", 1)[1])
    encoder = enc_cls(img_size=img_size, **enc_cfg.get("init_args", {}))

    net_cfg = config["model"]["init_args"]["network"]
    net_cls = getattr(importlib.import_module(net_cfg["class_path"].rsplit(".", 1)[0]),
                      net_cfg["class_path"].rsplit(".", 1)[1])
    net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k != "encoder"}
    network = net_cls(masked_attn_enabled=False, num_classes=num_classes,
                      encoder=encoder, **net_kwargs)

    lit_cfg = config["model"]
    lit_cls = getattr(importlib.import_module(lit_cfg["class_path"].rsplit(".", 1)[0]),
                      lit_cfg["class_path"].rsplit(".", 1)[1])
    model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}

    if "stuff_classes" in data_args:
        model_kwargs["stuff_classes"] = data_args["stuff_classes"]

    model = lit_cls(img_size=img_size, num_classes=num_classes,
                    network=network, **model_kwargs).eval().to(device)
    model.load_state_dict(state_dict, strict=False)

    return model, num_classes

@torch.no_grad()
def infer_semantic(model, img, device):
    imgs = [img.to(device)]
    img_sizes = [img.shape[-2:] for img in imgs]

    device_type = 'cuda' if device.type == 'cuda' else 'cpu'
    dtype = torch.float16 if device_type == 'cuda' else torch.bfloat16

    with autocast(device_type=device_type, dtype=dtype):
        if hasattr(model, 'window_imgs_semantic'):
            crops, origins = model.window_imgs_semantic(imgs)
            mask_logits_per_layer, class_logits_per_layer = model(crops)
            mask_logits = F.interpolate(mask_logits_per_layer[-1], model.img_size, mode="bilinear")
            crop_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits_per_layer[-1])
            logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        else:
            transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)
            mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)
            mask_logits = F.interpolate(mask_logits_per_layer[-1], model.img_size, mode="bilinear")
            prob_masks = mask_logits.sigmoid()
            prob_classes = class_logits_per_layer[-1].softmax(dim=-1)[..., :-1]
            semantic_logits = torch.einsum("bqc,bqhw->bchw", prob_classes, prob_masks)
            logits = model.revert_resize_and_pad_logits_instance_panoptic(semantic_logits, img_sizes)

    return logits[0].float().cpu()

def main():
    parser = ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument('--config', required=True)
    parser.add_argument('--ckpt', required=True)
    parser.add_argument('--method', default="maxlogit", choices=["maxlogit", "msp", "entropy", "rba"])
    parser.add_argument('--num-classes', type=int, default=None,
                        help="Override num_classes from config (e.g. 133 for COCO checkpoint)")
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[*] Running inference on: {device.type.upper()}")

    model, num_classes = load_eomt_model(args.config, args.ckpt, device,
                                         num_classes_override=args.num_classes)

    anomaly_score_list = []
    ood_gts_list = []

    for path in sorted(glob.glob(os.path.expanduser(args.input))):
        img = Image.open(path).convert('RGB')
        img = img.resize((1024, 512), Image.BILINEAR)
        img_t = torch.from_numpy(np.array(img)).permute(2, 0, 1)

        logits = infer_semantic(model, img_t, device).numpy()

        if args.method == "maxlogit":
            anomaly_result = 1.0 - np.max(logits, axis=0)
        elif args.method in ("msp", "rba"):
            probs = torch.softmax(torch.from_numpy(logits), dim=0).numpy()
            anomaly_result = 1.0 - np.max(probs, axis=0)
        elif args.method == "entropy":
            probs = torch.softmax(torch.from_numpy(logits), dim=0).numpy()
            entropy = -np.sum(probs * np.log(probs + 1e-10), axis=0)
            anomaly_result = entropy / np.log(num_classes)

        pathGT = path.replace("images", "labels_masks")
        if "RoadObsticle21" in pathGT: pathGT = pathGT.replace("webp", "png")
        if "fs_static"      in pathGT: pathGT = pathGT.replace("jpg",  "png")
        if "RoadAnomaly"    in pathGT: pathGT = pathGT.replace("jpg",  "png")

        mask = target_transform(Image.open(pathGT))
        ood_gts = np.array(mask)

        if "RoadAnomaly" in pathGT:
            ood_gts = np.where((ood_gts == 2), 1, ood_gts)
        if "LostAndFound" in pathGT:
            ood_gts = np.where((ood_gts == 0), 255, ood_gts)
            ood_gts = np.where((ood_gts == 1), 0,   ood_gts)
            ood_gts = np.where((ood_gts > 1) & (ood_gts < 201), 1, ood_gts)

        if 1 not in np.unique(ood_gts):
            continue

        ood_gts_list.append(ood_gts)
        anomaly_score_list.append(anomaly_result)

    if len(ood_gts_list) == 0:
        return

    ood_gts      = np.array(ood_gts_list)
    anomaly_scores = np.array(anomaly_score_list)

    ood_mask = (ood_gts == 1)
    ind_mask = (ood_gts == 0)

    val_out   = np.concatenate((anomaly_scores[ind_mask], anomaly_scores[ood_mask]))
    val_label = np.concatenate((np.zeros(np.sum(ind_mask)), np.ones(np.sum(ood_mask))))

    prc_auc = average_precision_score(val_label, val_out)
    fpr     = fpr_at_95_tpr(val_out, val_label)

    print(f'AUPRC score: {prc_auc*100.0:.2f}%')
    print(f'FPR@TPR95:   {fpr*100.0:.2f}%')

if __name__ == '__main__':
    main()

Overwriting /content/MaskArchitectureAnomaly_CourseProject/eval/run_eomt_anomaly.py


In [ ]:
import os

datasets = ["RoadAnomaly21", "RoadObsticle21", "FS_LostFound_full", "fs_static", "RoadAnomaly"]
methods  = ["maxlogit", "msp", "entropy", "rba"]

CONFIG_PATH     = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
CHECKPOINT_PATH = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco.bin"

%cd /content/MaskArchitectureAnomaly_CourseProject/eval/

print("=" * 50)
print(f"Testing Checkpoint: {CHECKPOINT_PATH.split('/')[-1]}")
print("=" * 50)

for dataset in datasets:
    print(f"\n\n{'='*40}\n>>> DATASET: {dataset} <<<\n{'='*40}")
    for method in methods:
        print(f"\n--- Method: {method.upper()} ---")
        input_path = f"Validation_Dataset/{dataset}/images/*.*"

        !python run_eomt_anomaly.py \
            --input "{input_path}" \
            --config "{CONFIG_PATH}" \
            --ckpt "{CHECKPOINT_PATH}" \
            --method {method} \
            --num-classes 133

/content/MaskArchitectureAnomaly_CourseProject/eval
Testing Checkpoint: eomt_coco.bin


>>> DATASET: RoadAnomaly21 <<<

--- Method: MAXLOGIT ---
[*] Running inference on: CUDA
[*] Inferred img_size=[640, 640] from checkpoint pos_embed.
[*] num_classes overridden to 133
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.
AUPRC score: 35.16%
FPR@TPR95:   88.07%

--- Method: MSP ---
[*] Running inference on: CUDA
[*] Inferred img_size=[640, 640] from checkpoint pos_embed.
[*] num_classes overridden to 133
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.
AUPRC score: 3